# Reviews

Address comments.

In [43]:
import ee
import geemap
from utils import *
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project = 'extents-490617')

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder

In [45]:
# there are objects defined in scripts 1 - 8 that will be used here. This requires running them in this script:

import nbimporter # lets you import notebooks like regular modules

%run 3_grids.ipynb # run full script here
# %run 6_mature.ipynb # run full script here
# %run 7_write_csv.ipynb # run full script here
# %run objects.ipynb

In [5]:
edge = ee.Image(f"{data_folder}/distance_to_secondary_edge").gt(30).rename("edge") # binary mask to identify edge pixels



# --- 1. Get projections ONCE outside the loop ---
sd_proj = biomass.projection()
age_proj = age.projection()

# Get SD pixel coordinates at SD resolution
sd_coords = ee.Image.pixelLonLat().reproject(sd_proj)

# Get age pixel coordinates at age resolution  
age_coords = ee.Image.pixelLonLat().reproject(age_proj)

# Reproject SD coords to age resolution (nearest neighbor = snap to parent center)
sd_lon_at_age = sd_coords.select('longitude').reproject(age_proj)
sd_lat_at_age = sd_coords.select('latitude').reproject(age_proj)

# Distance in degrees between age pixel center and its parent SD pixel center
d_lon = age_coords.select('longitude').subtract(sd_lon_at_age)
d_lat = age_coords.select('latitude').subtract(sd_lat_at_age)

# Convert to meters (approximate, valid near equator — fine for Amazon)
meters_per_deg_lon = ee.Image.constant(111320).multiply(
    age_coords.select('latitude').multiply(3.141592653589793 / 180).cos()
)
meters_per_deg_lat = ee.Image.constant(110540)

dx = d_lon.multiply(meters_per_deg_lon).abs()
dy = d_lat.multiply(meters_per_deg_lat).abs()

inner_mask = dy.lte(35).And(dx.lte(35))

over_1ha_features = ee.FeatureCollection("projects/forestregrowth/assets/secondary_polygons/secondary_age_vectors_022")



inner_patch = ee.Image(0)

proj = age.projection().getInfo()

# age = age.clip(amazon)
for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 1, kernelType="square", units='pixels')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask().reproject(age.projection())






# map = geemap.Map()
# map.addLayer(edge.updateMask(age), {'min':0, 'max':1, 'palette':['red', 'blue']}, 'edge')
# # map.addLayer(biomass, {'min':100, 'max':300, 'palette':['red', 'blue']}, 'biomass')
# # map.addLayer(inner_mask, {'min':0, 'max':1, 'palette':['black', 'white']}, 'inner_mask')
# # map.addLayer(over_1ha_features, {}, 'over_1ha_features')
# # map.addLayer(age, {'min':1, 'max':35, 'palette':['red', 'blue']}, 'age')
# # map.addLayer(inner_patch, {}, 'inner_patch')
# map

## GEDI - mean biomass per 10km grid cell

Aggregating GEDI L4A into 10km pixels doesn't work as fast and as well with reduceResolution + reproject (as done on 6_mature.ipynb). That process ends up running out of computational power.

That happens because GEDI data needs to be aggregated over the years, which can be computationally costly, and because reprojecting to mask over the mature_mask also takes a lot of computational power.

In order to avoid this, it is necessary to get the mean per 10km grid cell over many tiles.


In [46]:
amazon = ee.FeatureCollection("projects/forestregrowth/assets/raw/biomes_br").filter(ee.Filter.eq('CD_Bioma', 1))

def quality_mask(image):
    image = image.updateMask(image.select('l4_quality_flag').eq(1)) \
              .updateMask(image.select('degrade_flag').eq(0))
    relative_se = image.select('agbd_se').divide(image.select('agbd'))
    return image.updateMask(relative_se.lte(0.5))

GEDI = (ee.ImageCollection('LARSE/GEDI/GEDI04_A_002_MONTHLY')
        .filterDate('2019-01-01', '2020-12-31')
        .filterBounds(amazon)
        .map(quality_mask)
        .select(['agbd']))

GEDI = GEDI.mosaic().setDefaultProjection(age.projection()).rename('GEDI_biomass')

GEDI_mature = GEDI.updateMask(mature_mask).rename("GEDI_mature_biomass")


In [47]:
#get only gedi pixels contained within continuous patches of secondary forest age.
inner_patch = ee.Image(0)
proj = age.projection().getInfo()

# age = age.clip(amazon)
for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 1, kernelType="square", units='pixels')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask().reproject(age.projection())

### Grid sampling - GEDI

Use the same sampling method to obtain one pixel of secondary forest that coincides with a GEDI footprint per 10km2

Objects imported from 3_grids.ipynb


In [48]:
GEDI_reproj = GEDI.updateMask(inner_patch).rename("GEDI_biomass")

create_grid(GEDI_reproj, region_name = "amazon", cell_size = 10000, file_name = "secondary_edge_removed_gedi")

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 200000)  # 200km tiles

# Convert to a list so we can iterate
grid_list = grid.toList(grid.size())
n_tiles = grid.size().getInfo()
print(f"Number of tiles: {n_tiles}")

for i in range(n_tiles):
    tile = ee.Feature(grid_list.get(i)).geometry()
    
    # Aggregate within this tile only
    tile_image = GEDI_mature \
        .setDefaultProjection(crs='EPSG:4326', scale = 25) \
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=1024,
            bestEffort=True
        ) \
        .reproject(crs='EPSG:4326', scale = 10000) \
        .clip(tile)
    
    task = ee.batch.Export.image.toAsset(
        image=tile_image,
        description=f"GEDI_mature_biomass_10k_tile_{i}",
        assetId=f"{data_folder}/GEDI_mature/GEDI_mature_biomass_10k_tile_{i}",
        region=tile,
        crs='EPSG:4326',
        scale=10000,
        maxPixels=1e13
    )
    # task.start()
    # print(f"Started tile {i}/{n_tiles}")


### GEDI nearest neighbor
After the GEDI data is aggregated to 10km resolution, we use the same method from 6_mature to get the nearest mature biomass for the gaps in the image (areas where there was no GEDI read on mature forests in the 10km grid cell pixel)

In [ ]:
all_images = import_folder_features(f"{data_folder}/GEDI_mature", asset_type='image')
mature_biomass_10k = ee.ImageCollection(all_images).mosaic()

features_secondary = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed")

# obtain_nearest_mature_neighbor(mature_biomass_10k, features_secondary, "_GEDI")

In [ ]:
grid_gedi = ee.FeatureCollection(f"{data_folder}/grid_10k_amazon_secondary_edge_removed_gedi")

nearest_mature_GEDI = ee.Image(f"{data_folder}/nearest_mature_GEDI").rename("nearest_mature_GEDI")

unified_data_secondary = ee.Image.cat([unified_data, age, GEDI_reproj, nearest_mature_GEDI, reduce_reproject(biomass, ee.Reducer.mean()).rename("ESA_biomass")])

# export_csv("secondary_GEDI_ESA", unified_data_secondary, 10, n_chunks = 30, grid = grid_gedi)

## Same-age patches

Keep only patches of the same age and greater than 1ha

Do not exclude edge pixels. Selecting exclusively based on area and age.

In [ ]:
grid = amazon.geometry().coveringGrid('EPSG:4326', 100000)

grid_list = grid.toList(grid.size())
n = grid.size().getInfo()

for i in range(n):
    cell = ee.Feature(grid_list.get(i))

    vectors = age.reduceToVectors(
        geometry=cell.geometry(),
        geometryType='polygon',
        scale=30,
        eightConnected=True,
        maxPixels=1e12,
        labelProperty='age',
        tileScale = 16
    )

    vectors = vectors.map(
        lambda f: f.set({
            'area_m2': f.geometry().area(maxError=1),
            'tile_id': i + 1
        })
    ).filter(ee.Filter.gt('area_m2', 10000)).select(['age', 'tile_id'])

    task = ee.batch.Export.table.toAsset(
        collection=vectors,
        description=f"secondary_age_vectors_{i+1:03d}",
        assetId=f"{data_folder}/secondary_polygons/secondary_age_vectors_{i+1:03d}"
    )
    # task.start()

#309 needs to be run again

### Export age and biomass for ESA CCI for the same-age patches

In [ ]:
# for each collection, we will make a new collection by sampling one random pixel of age within it and returning it as a point feature.

def one_point_per_poly(f):
    geom = f.geometry()
    pt = ee.FeatureCollection.randomPoints(
        region = geom,
        points = 1,
        seed = ee.Number(1),   # or derive a deterministic seed if needed
        maxError = 1
    ).first()
    return ee.Feature(pt).copyProperties(f)

asset_list = ee.data.listAssets({'parent': f"{data_folder}/secondary_polygons"})['assets']

feature_ids = [a['name'] for a in asset_list]                    

for asset_id in feature_ids:
    polygons_fc = ee.FeatureCollection(asset_id)

    points_fc = ee.FeatureCollection(polygons_fc.map(one_point_per_poly))

    task = ee.batch.Export.table.toAsset(
        collection = points_fc,
        description = f"secondary_age_points_{asset_id[-3:]}",
        assetId = f"{data_folder}/secondary_points/secondary_age_vectors_{asset_id[-3:]}"
    )
    # task.start()


Try to get only patches with 9x9 same age

In [21]:
inner_patch = ee.Image(0)

proj = age.projection().getInfo()

age = age.clip(amazon)

for year in range(1, 35):
    age_img = age.eq(year).unmask(0)

    sum_neighbors = age_img.focalMin(radius = 30, kernelType="square", units='meters')

    inner_patch = inner_patch.add(sum_neighbors)

inner_patch = inner_patch.selfMask()

task = ee.batch.Export.image.toAsset(
    image=inner_patch,
    description="inner_patch",
    assetId=f"{data_folder}/inner_patch",
    region=roi,
    crs=proj['crs'],
    crsTransform=proj['transform'],
)
task.start()